# Concatenate Spectractor Results from Corentins Runs

- author : Sylvie Dagoret-Campagne
- creation date : 2026-03-15
- last update : 2026-03-15 : run2026_v02a,b,c,d,e,f,g  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

plt.rcParams["figure.figsize"] = (16,8)
plt.rcParams["axes.labelsize"] = 'xx-large'
plt.rcParams['axes.titlesize'] = 'xx-large'
plt.rcParams['xtick.labelsize']= 'xx-large'
plt.rcParams['ytick.labelsize']= 'xx-large'
plt.rcParams["legend.fontsize"] = "xx-large"


In [ ]:
# CHECK THE CONFIG HERE !!!!!!
from BUTLER00_parameters import *

In [ ]:
DumpConfig()

In [ ]:
# Enable interactive matplotlib backend with zoom/pan toolbar
# Requires: pip install ipympl
# If ipympl is not available, fall back to inline (no interactivity)
try:
    import ipympl  # noqa: F401
    %matplotlib widget
    print("ipympl found → interactive backend (%matplotlib widget)")
except ImportError:
    %matplotlib inline
    print("ipympl NOT found → falling back to %matplotlib inline (no zoom widget)")
    print("Install with:  pip install ipympl")


## Configuration

### Define old/ recent data

In [ ]:
selected_run_olddata = "run2026_v02b_cr"
#selected_run_recentdata = "run2026_v02c_cr"
selected_run_recentdata =  "run2026_v02d_cr" 

In [ ]:
selected_run = [ selected_run_olddata, selected_run_recentdata]

In [ ]:
atmfilename_olddata = extractedfilesdict[selected_run_olddata]
tag_olddata = selected_run_olddata 
input_path_olddata = os.path.dirname(atmfilename_olddata)
inputfilename_olddata = os.path.basename(atmfilename_olddata)

In [ ]:
atmfilename_recentdata = extractedfilesdict[selected_run_recentdata]
tag_recentdata = selected_run_recentdata 
input_path_recentdata = os.path.dirname(atmfilename_recentdata)
inputfilename_recentdata = os.path.basename(atmfilename_recentdata)

In [ ]:
print(f"Spectractor Extracted atmospheric parameters file Old Data : {atmfilename_olddata}  for config {tag_olddata}")
print(f"Spectractor Extracted atmospheric parameters file Recent Data : {atmfilename_recentdata}  for config {tag_recentdata}")

## Prepare Output

In [ ]:
tag_finaldata = tag_olddata + "_" + tag_recentdata
final_filename = "auxtel_atmosphere_feb26_gaiaspec" + "_" + tag_finaldata + ".npy"
print(f"final tag :: {tag_finaldata} \t ==> \t final filename :: {final_filename} ")

## Read Old /Recent dataframes

In [ ]:
if "run2026_v01" not in selected_run: 
    specdata_olddata = np.load(atmfilename_olddata,allow_pickle=True)
    df_spec_olddata = pd.DataFrame(specdata_olddata)
    specdata_recentdata = np.load(atmfilename_recentdata,allow_pickle=True)
    df_spec_recentdata = pd.DataFrame(specdata_recentdata)
else:
    df_spec_olddata = pd.read_parquet(atmfilename_olddata)
    df_spec_recentdata = pd.read_parquet(atmfilename_recentdata)

In [ ]:
df_spec_olddata.head()

In [ ]:
df_spec_recentdata.head()

In [ ]:
print(f"recent data \t  ==>  {tag_recentdata} \t ::  (tmin,tmax)",df_spec_recentdata["DATE-OBS"].min()," , " ,df_spec_recentdata["DATE-OBS"].max())
print(f"recent data \t  ==>   {tag_recentdata}\t ::  N = {len(df_spec_recentdata)}") 
print(f"old data    \t ==>  {tag_olddata}    \t ::  (tmin,tmax)",df_spec_olddata["DATE-OBS"].min()," , " ,df_spec_olddata["DATE-OBS"].max())
print(f"old data    \t  ==>   {tag_olddata}\t ::  N = {len(df_spec_olddata)}") 

## Really concatenate

In [ ]:
df_spec_concat = (
    pd.concat([df_spec_recentdata, df_spec_olddata], ignore_index=True)
      .drop_duplicates(subset="DATE-OBS", keep="first")
      .reset_index(drop=True)
)

In [ ]:
print(f"new data \t  ==>   {tag_finaldata}\t ::  (tmin,tmax)",df_spec_concat["DATE-OBS"].min()," , " ,df_spec_concat["DATE-OBS"].max())
print(f"new data \t  ==>   {tag_finaldata}\t ::  N = {len(df_spec_concat)}") 

## Save Data

In [ ]:
if "2026_v01" not in selected_run: 
    rec_array = df_spec_concat.to_records(index=False)
    np.save(final_filename , rec_array)
else:
    df_spec_concat.to_parquet(final_filename, compression='gzip')

## Check readability

In [ ]:
if "2026_v01" not in selected_run: 
    newspecdata = np.load(final_filename ,allow_pickle=True)
    df_newspec = pd.DataFrame(newspecdata)
else:
    df_newspec = pd.read_parquet(final_filename )

In [ ]:
#newspecdata

In [ ]:
df_newspec.head()

In [ ]:
df_newspec.tail()

In [ ]:
print(f"new data \t  ==>   {tag_finaldata}\t ::  (tmin,tmax)",df_newspec["DATE-OBS"].min()," , " ,df_newspec["DATE-OBS"].max())
print(f"new data \t  ==>   {tag_finaldata}\t ::  N = {len(df_newspec)}") 